# ODOT Roadway Data

In [4]:
import folium
import requests
import geopandas
from shapely.geometry import Polygon # <-- Import the Polygon class explicitly

# 1. DEFINE THE CORRECT ENDPOINT
url = "https://tims.dot.state.oh.us/ags/rest/services/Boundaries/County/MapServer/0/query"

# Parameters
params = {
    'where': '1=1',
    'outFields': '*',
    'returnGeometry': 'true',
    'f': 'json',
    'outSR': '4326'
}

# Headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

print("Fetching county data from the ODOT endpoint...")
gdf = None

# 2. FETCH AND PROCESS THE DATA MANUALLY
try:
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    if 'features' in data and data['features']:
        
        cleaned_data = []
        for feature in data["features"]:
            if feature.get('geometry') and feature.get('geometry').get('rings'):
                # ==> THE FINAL FIX <==
                # Manually construct the Polygon object from the "rings" data.
                # The first ring is the outer boundary (shell).
                # Subsequent rings are holes in the polygon.
                rings = feature['geometry']['rings']
                shapely_geom = Polygon(rings[0], rings[1:])
                
                # Extract only the attributes we need
                attributes = feature.get('attributes', {})
                cleaned_record = {
                    'geometry': shapely_geom,
                    'COUNTY_NAME': attributes.get('COUNTY_NAME'),
                    'CNTY_SEAT_NAME': attributes.get('CNTY_SEAT_NAME'),
                    'DISTRICT': attributes.get('DISTRICT')
                }
                cleaned_data.append(cleaned_record)
        
        if cleaned_data:
            gdf = geopandas.GeoDataFrame(cleaned_data, crs="EPSG:4326")
            print(f"Data processed successfully! Found {len(gdf)} valid counties.")
        else:
            print("Warning: No valid county features could be processed.")

    else:
        print("Error: The server responded, but did not return any map features.")
        print("Server message:", data)

except requests.exceptions.RequestException as e:
    print(f"An error occurred during the HTTP request: {e}")
    if 'response' in locals():
        print("Server responded with:", response.text[:500])

# 3. CREATE THE MAP
if gdf is not None and not gdf.empty:
    m = folium.Map(location=[40.4, -82.9], zoom_start=7, tiles="CartoDB positron")
    style_function = lambda x: {'fillColor': '#a6cee3', 'color': '#1f78b4', 'weight': 1.5, 'fillOpacity': 0.6}

    # 4. CREATE THE LAYER
    for _, row in gdf.iterrows():
        popup_html = f"""
        <b>County:</b> {row['COUNTY_NAME']}<br>
        <b>County Seat:</b> {row['CNTY_SEAT_NAME']}<br>
        <b>ODOT District:</b> {row['DISTRICT']}
        """
        iframe = folium.IFrame(popup_html, width=250, height=80)
        popup = folium.Popup(iframe)
        tooltip = row['COUNTY_NAME']
        
        folium.GeoJson(
            row.geometry,
            style_function=style_function,
            tooltip=tooltip,
            popup=popup
        ).add_to(m)

    # 5. SAVE THE MAP
    output_file = 'ohio_county_map_final.html'
    m.save(output_file)
    print(f"\nMap has been generated and saved to '{output_file}'.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\site-packages\traitlets\config\appl

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [12]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

# 1. Connect anonymously for public data
gis = GIS()

# 2. Define the URL to the Feature Layer
county_layer_url = "https://tims.dot.state.oh.us/ags/rest/services/Boundaries/County/MapServer/0"

# 3. Create a FeatureLayer object
county_layer = FeatureLayer(county_layer_url)

# 4. Create a map centered on Ohio
map_view = gis.map('Ohio')
map_view.zoom = 7

# Add the data layer to the map using the older 'add_layers' method (plural)
map_view.add_layers(county_layer)

# 5. Display the map in your Jupyter Notebook
map_view


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\dparks1\AppData\Local\anaconda3\envs\esri\Lib\site-packages\traitlets\config\appl

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



AttributeError: 'Map' object has no attribute 'add_layers'